# Gigavolt Milestone 4 — measured vector retrieval
This template has no invented outputs. Execute it after the tuning and held-out tests. The held-out result is the final result, even if it misses a target.

Database retrieval uses a warm persistent local connection and includes fetching five passages. It excludes embedding generation, model loading, and display. ANN recall compares neighbors with exhaustive vector search; it is not a measure of answer correctness.

In [ ]:
import os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
run = Path(os.environ.get('GIGAVOLT_RUN', '../runs/run01')).resolve()
def read(name): return json.loads((run/name).read_text())
inventory = read('inventory.json')
chunk_report = read('chunk_report.json')
embedding_report = read('embedding_report.json')
method = read('benchmark_test/method.json')
selection = read('selection.json')
print('Run:', run)
print('PDFs:', inventory['pdf_count'], 'pages:', inventory['pdf_pages'])
print('Service records:', inventory['service_records'])
print('Chunk report:', chunk_report)
print('Embedding report:', embedding_report)
print('MariaDB:', method['mariadb'])
print('Method:', method['timing'])
print('Selected using tuning only:', selection)
tune = pd.read_csv(run/'benchmark_tune/summary.csv')
test = pd.read_csv(run/'benchmark_test/summary.csv')
raw = pd.read_csv(run/'benchmark_test/raw.csv')
builds = pd.DataFrame(read('builds.json'))
display(tune)
display(test)
display(builds)

## Held-out latency distribution
These are repeated requests across ten fixed questions, not hundreds of independently sampled questions. Percentiles describe this experiment and hardware; tail estimates from 200 requests are limited.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.hist(raw.latency_ms, bins=25, color='#2463a2', edgecolor='white')
ax.axvline(20, color='#be3434', linestyle='--', label='20 ms target')
for column, color in [('p50_ms','#157f45'),('p95_ms','#9a6200'),('p99_ms','#753a99')]:
    ax.axvline(test.iloc[0][column], color=color, label=f"{column}: {test.iloc[0][column]:.3f}")
ax.set(xlabel='Database retrieval time (ms)', ylabel='Requests', title='Held-out retrieval latency')
ax.legend(); fig.tight_layout(); fig.savefig(run/'latency_distribution.png',dpi=180)
plt.show()

## Tuning trade-off
The displayed recall is strict top-five chunk-ID agreement with an exhaustive comparison. Tie-aware recall is retained separately in the summary. M changes graph connectivity; search effort changes how much of that graph is explored. MariaDB does not expose a separately documented ef_construction setting.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for m, group in tune.groupby('M'):
    ax.scatter(group.p50_ms,group.recall5,label=f'M={m}',s=65)
    for _,r in group.iterrows(): ax.annotate(f"ef={int(r.ef_search)}",(r.p50_ms,r.recall5),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.axvline(20,color='gray',linestyle='--'); ax.axhline(.9,color='gray',linestyle='--')
ax.set(xlabel='Median database retrieval time (ms)',ylabel='Strict recall@5',title='Tuning set: speed and recall',ylim=(0,1.06))
ax.legend(); fig.tight_layout(); fig.savefig(run/'recall_latency.png',dpi=180); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(builds.M.astype(str),builds.load_and_index_seconds,color='#2463a2')
ax.set(xlabel='M',ylabel='Seconds',title='Combined insertion and index maintenance')
fig.tight_layout(); fig.savefig(run/'load_index_time.png',dpi=180); plt.show()
r = test.iloc[0]
passed = r.p50_ms < 20 and r.recall5 > .90
print('Held-out target met:', bool(passed))
print(f"P50={r.p50_ms:.3f} ms; P95={r.p95_ms:.3f} ms; P99={r.p99_ms:.3f} ms")
print(f"Strict recall@5={r.recall5:.4f}; tie-aware recall@5={r.tie_aware_recall5:.4f}")
print('Do not replace this statement with the best individual query timing.')

## Human review — complete after executing
Add at least ten question/result examples, including service-record queries. For each include the query, chunk ID, source and page, relevance judgment, and reason. Do not treat the source hints as complete answer labels.

## Interpretation — complete using actual results
Explain why the selected setting was chosen, whether the held-out targets were met, what changed between settings, and any failures. Record the instance type, RAM, CPU, disk, database version and dependency lock file. Describe the data volume without claiming large-scale performance.

## Limitations
- Small supplied document collection; service-record volume depends on the real export.
- PDF text extraction can lose table relationships; pages remain separate.
- Model is a compact general English baseline; technical relevance needs human inspection.
- Single client, warmed caches, repeated fixed queries, no concurrency study.
- Database time is not end-to-end response time.
- Exact-vector recall is not factual correctness; ties are reported separately.
- Load/index timings include SQL insertion overhead and are collected sequentially.
- No independently exposed construction-effort parameter in the documented MariaDB interface.

## Sources
[MariaDB vectors](https://mariadb.com/docs/server/reference/sql-structure/vectors/vector-overview), [vector settings](https://mariadb.com/docs/server/reference/sql-structure/vectors/vector-system-variables), [embedding model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).